In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

**Step 1 - Read bronze `circuits` table**

In [0]:
circuits_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
display(circuits_df)

**Step 2 - Keep only the columns required (Drop url column)**

In [0]:
circuits_selected_df = circuits_df.select("circuitId",
                   "circuitName",
                   "lat",
                   "long",
                   "locality",
                   "country",
                   "ingestion_timestamp",
                   "source_file",
                   "batch_id"
                   )

**Step 3 & 4 - Standardise Column Names**

In [0]:
# circuits_renamed_df = (
#    circuits_selected_df
#        .withColumnRenamed("circuitId", "circuit_id")
#        .withColumnRenamed("lat", "latitude")
#        .withColumnRenamed("long", "longitude")
#        .withColumnRenamed("circuitName", "circuit_name")
#)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({"circuitId":"circuit_id",
                            "lat":"latitude",
                            "long":"longitude",
                            "circuitName":"circuit_name"
        })
)

**Step 5 - Filter out rows where circuit_id is null (business key validation)**

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

**Step 6 - Remove duplicate records**

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

**Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case**

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))
)

**Step 8 - Write the transformed data to silver `circuits` table**

In [0]:
# circuits_final_df = (
#    circuits_final_df
#        .withColumn("created_timestamp", F.current_timestamp())
#        .withColumn("updated_timestamp", F.current_timestamp())
# )

In [0]:
# if not spark.catalog.tableExists(silver_table):
#    (
#        circuits_final_df
#            .write
#            .format('delta')
#            .mode('overwrite')
#            .saveAsTable(silver_table)
#    )

# else:
#    from delta.tables import DeltaTable
#
#    delta.table = DeltaTable.forName(spark, silver_table)
#
#    (
#        delta_table.alias("t")
#        .merge(
#            circuits_final_df.alias("s"),
#            "t.circuit_id = s.circuit_id"
#        )
#        .whenMatchedUpdate(
#            condition = "s.batch_id" >= "t.batch_id",
#            set = {
#                "circuit_name": "s.circuit_ref",
#                "lantitude": "s.lantitude",
#                "longtitude": "s.longtitude",
#                "locality": "s.locality",
#                "country": "s.country",
#                "ingestion_timestamp": "s.ingestion_timestamp",
#                "source_file": "s.source_file",
#                "batch_id": "s.batch_id",
#                "updated_timestamp": "s.updated_timestamp",
#            }
#        )
#        .whenNotMatchedInsertAll()
#        .execute()
#    )

In [0]:
write_to_silver(
    input_df =  circuits_final_df,
    target_table = silver_table,
    merge_condition = "t.circuit_id = s.circuit_id",
    columns_to_update = [
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
select * from formula1_incr.silver.circuits